`codes` is a `(n_samples, n_basis)` array of sparse coefficients. Each column corresponds to a basis vector; each value indicates how strongly that basis vector was activated for a given sample.

Coefficients are continuous and peak sharply at zero, so we need a threshold to decide when a basis vector is meaningfully active (i.e. its coefficient is non-zero).

Two thresholding methods are compared:
- **GMM (2 components)**: fits a Gaussian to the zero-peak and another to the active population; the threshold is their boundary.
- **Otsu**: variance-based baseline; works reasonably well given the bimodal structure but less principled.

The core thresholding logic works as follows: for a given basis vector's coefficients, we split values into positive and negative sides and apply threshold_fn to each independently. The sign is then inferred by checking whether each side's maximum falls within the other side's garbage threshold — if the negative side's maximum is smaller than the positive threshold, the negative values are just noise and the data is non-negative, and vice versa. This produces one of four outcomes: NonNegative, NonPositive, Ambiguous (both sides are noise), or MixedSign (both sides have real activations).  

The thresholding flags cases where the data cannot be cleanly classified as non-negative or non-positive. On inspection, these correspond to dead atoms — basis vectors never meaningfully activated across any sample.


Empirically, we now have sufficient proof to be concerned when the coefficients are neither non-negative nor non-positive.  
It's best to flag it and analyse them manually. For other cases, we automatically find thresholds.  

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import random
import torch
from pt_to_api.mnist import SimpleMNIST
from pt_to_api.utils import show_single_channel_red_green_black as S, to_show_list as tsl
import gc
import pickle
from pt_to_api.utils import otsu_threshold
from sklearn.preprocessing import normalize
from collections import defaultdict
import math
from sklearn.cluster import HDBSCAN
from sklearn.metrics.pairwise import cosine_similarity
import json
from dataclasses import dataclass
from pt_to_api import benchmark as B
from sklearn.mixture import GaussianMixture

In [ ]:
MODEL_PATH = Path("../../../pt-to-api/data/model.pt")
INPUT_PATH = Path("../../../pt-to-api/data/first-input-tens.pt")

inp = torch.load(INPUT_PATH, weights_only=False)
model = SimpleMNIST()
model.load_state_dict(torch.load(MODEL_PATH))

In [ ]:
DRIVE_PATH = Path("/Users/hariomnarang/Desktop/gdrive-sync/hiccup/djl-on-mnist")
MAIN_OUT_DIR = (DRIVE_PATH / "collect-patches" / "data")

In [ ]:
SingleRun = B.SingleRun
Autoencoder = B.Autoencoder


@dataclass(frozen=True)
class RunId:
    n_components: int
    seed: int

@dataclass(frozen=True)
class CompId:
    run_id: RunId
    idx_inside_run: int

@dataclass(frozen=True)
class IdAndComp:
    comp_id: CompId
    comp: np.ndarray


C2R = dict[RunId, SingleRun]

def load_c2r(runs_dir: Path) -> C2R:
    c2r = {}
    for p in runs_dir.rglob("*.pt"):
        seed = int(p.stem.split("_")[1])
        n_components = int(p.parent.stem)
        c2r[RunId(n_components, seed)] = torch.load(p, weights_only=False)
    return c2r

We will use a single layer + channel combination for testing here. I'm not going to test it a lot for now.  
This notebook is however, a reminder that the testing should be done on other datasets too.  

In [ ]:
layer_name, channel = "layers.2", 0

layer_data_dir = MAIN_OUT_DIR / layer_name / str(channel)
runs_dir =  layer_data_dir / "runs"
SHAPE = (8,9)

c2r = load_c2r(runs_dir)

In [ ]:
from collections import defaultdict

n_components_list = sorted(set([c.n_components for c in c2r]))
n2seeds = defaultdict(list)
for c in c2r:
    n2seeds[c.n_components].append(c.seed)
n_components_list, n2seeds

In [ ]:
n_components_list, n2seeds[7]
run = c2r[RunId(7,0)]
S([run.components[3].reshape(SHAPE)])
plt.show()

In [ ]:
from typing import Literal
from dataclasses import dataclass
import numpy as np

@dataclass
class NonNegative:
    threshold: float

@dataclass
class NonPositive:
    threshold: float

@dataclass
class MixedSign:
    t_pos: float
    t_neg: float
    pos_max: float
    neg_max: float

@dataclass
class Ambiguous:
    pass

SignResult = NonNegative | NonPositive | MixedSign | Ambiguous


def find_threshold_using_gmm(values, conservatism=0):
    values = np.array(values)
    gmm = GaussianMixture(n_components=2, random_state=0)
    gmm.fit(values.reshape(-1, 1))
    
    garbage_idx = np.argmax(gmm.means_)
    real_idx = 1 - garbage_idx
    
    midpoint = (gmm.means_[garbage_idx] + gmm.means_[real_idx]) / 2
    gap = gmm.means_[real_idx] - gmm.means_[garbage_idx]  # signed, points away from garbage
    
    return (midpoint - conservatism * gap).item()

def check_and_threshold(values: np.ndarray, threshold_fn: callable) -> SignResult:
    """
    Determines the sign structure of the data and returns the appropriate threshold.

    The threshold_fn is called on absolute values. The sign is inferred by checking
    whether each side's max falls within the other side's garbage threshold:
    - If neg_max < pos_threshold: data is non-negative (negative values are noise)
    - If pos_max < neg_threshold: data is non-positive (positive values are noise)
    - If both: ambiguous (everything is near zero)
    - If neither: genuinely mixed sign

    Args:
        values: raw (signed) attribution values
        threshold_fn: callable that takes absolute values and returns a threshold (also in absolute value space)

    Returns:
        one of NonNegative, NonPositive, MixedSign, Ambiguous
    """
    values = np.array(values)
    pos = values[values > 0]
    neg = values[values < 0]

    t_neg = threshold_fn(np.abs(neg)) if len(neg) > 0 else 0
    t_pos = threshold_fn(pos) if len(pos) > 0 else 0

    pos_max = pos.max() if len(pos) > 0 else 0
    neg_max = abs(neg.min()) if len(neg) > 0 else 0

    is_non_neg = neg_max < t_pos
    is_non_pos = pos_max < t_neg

    if is_non_neg and is_non_pos:
        return Ambiguous()
    elif is_non_neg:
        return NonNegative(threshold=t_pos)
    elif is_non_pos:
        return NonPositive(threshold=-t_neg)
    else:
        return MixedSign(t_pos=t_pos, t_neg=t_neg, pos_max=pos_max, neg_max=neg_max)


def plot_comparison_grid(codes):
    n = codes.shape[1]
    cols = 4
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3))
    axes = axes.flatten()

    for i in range(n):
        # data = np.abs(codes[:, i])
        data = codes[:, i]
        t_otsu = otsu_threshold(data)
        t_gmm_res = check_and_threshold(data, find_threshold_using_gmm)
        t_gmm = None
        match t_gmm_res:
            case NonNegative(threshold=t): t_gmm = t
            case NonPositive(threshold=t): t_gmm = t
            case MixedSign(): 
                print(f"found GMM threshold on both positivve and negative side: comp={i} result={t_gmm_res}") 
            case Ambiguous(): 
                print(f"could not find any GMM threshold. Ambiguous. comp={i} result={t_gmm_res}")
        axes[i].hist(data, bins=50, color='steelblue', edgecolor='none')
        axes[i].axvline(t_otsu, color='red', linewidth=1.5, label=f'otsu={t_otsu:.3f}')
        if t_gmm is not None:
            axes[i].axvline(t_gmm, color='orange', linewidth=1.5, label=f'gmm={t_gmm:.3f}')
        axes[i].legend(fontsize=7)
        axes[i].set_title(f'component {i}', fontsize=9)

    for j in range(n, len(axes)):
        axes[j].set_visible(False)

    plt.tight_layout()
    plt.show()

In [ ]:
def find_thresholds_for_each_component(codes):
    threshes = []
    for i in range(codes.shape[1]):
        data = codes[:, i]
        t_gmm_res = check_and_threshold(data, find_threshold_using_gmm)
        t_gmm = None
        match t_gmm_res:
            case NonNegative(threshold=t): t_gmm = t
            case NonPositive(threshold=t): t_gmm = t
            case MixedSign(): 
                print(f"WARN: found GMM threshold on both positivve and negative side: comp={i} result={t_gmm_res}") 
            case Ambiguous(): 
                print(f"WARN: could not find any GMM threshold. Ambiguous. comp={i} result={t_gmm_res}")
        threshes.append(t_gmm)
    return threshes

# Comparing Otsu and GMM

For a single dataset, I've trained an autoencoder for each (components, seed) combination. `c2r` contains those.   
The values contain the `codes` array.  

Each histogram below is the `codes` array for each component in the autoencoder. We generally see two types of distributions:
- unimodal peak at 0
- bimodal peak at 0 and at some other value.  


We check two thresholding methods:
- otsu 
- 2 clusters GMM, using the midpoint of means as threshold

Otsu is good for bimodal data. It can overshoot for unimodal. GMM has reasonable performance in both.  

In [ ]:
for c in n_components_list:
    seed = 0
    run = c2r[RunId(c, seed)]
    print(f"############################# num_components: {c} ########################################")
    plot_comparison_grid(run.codes)

# GMM failure

We have some cases in the above graph (You need to peek at n_components=15 case).  
We have a warning saying that GMM could not find a non-negative or a non-positive threshold.  
It would be useful to investigate this. I'll call it the GMM mixed sign case.  

We first instantiate the thresholds using GMM.  
The warning says that component `9` has the problem.  

In [ ]:
def show_comps_activity_for_case(codes, components, recon, thresholds, sample_idx, resize_shape):
    """This function shows the weighted components (code[i]*component[i]) for a given sample at `sample_idx`.

    The function shows the relative value of the coefficient of each component compared to the others
    It also shows whether the component is marked as active or not
    """
    _codes = codes[sample_idx]
    active_stats = []
    for j in range(len(_codes)):
        t = thresholds[j]
        if t is None:
            active_stats.append("unknown")
        elif np.abs(_codes[j]) > np.abs(t):
            active_stats.append("active")
        else:
            active_stats.append("inactive")

    comps = [_codes[i]*components[i].reshape(resize_shape) for i in range(len(_codes))]
    maxes = [np.abs(c).max() for c in comps]
    maxes /= np.max(maxes)
    cols = min(8, len(components))
    rows = math.ceil(len(components) / cols)
    S(comps, (20,rows*3), cols, ax_titles=[f"{m:.3f}/{a}" for (m,a) in zip(maxes, active_stats)], suptitle="weighted components")
    plt.show()
    S([recon[sample_idx].reshape(resize_shape)], (3,2), suptitle="reconstruction")
    plt.show()

In [ ]:
run = c2r[RunId(15,0)]
thresholds = find_thresholds_for_each_component(run.codes)

# Investigating component 9 (total components=15)

We find the index of the max coefficient of component 9. You'll notice that it looks very weak compared to others. Considering that it is its maximum case, this is a dead atom.  


In [ ]:
# lets check out the components with the max value of the coefficient for comp=9
max_idx= np.argmax(np.abs(run.codes[:, 9]))
show_comps_activity_for_case(run.codes, run.components, run.recon, thresholds, max_idx, SHAPE)

Dead atoms are bad news, below we have the loss plots. We notice that the loss has started going upwards around 15.  
So the threshold is fine. It is also acting as a warning flag for badly behaving components.  

In [ ]:
comps_list = sorted(set([c.n_components for c in c2r]))
losses = []
for n in comps_list:
    min_loss_of_comp = np.min([c2r[RunId(n, seed)].loss for seed in [0,1,2]])
    losses.append(min_loss_of_comp)
plt.plot(comps_list, losses)
plt.show()

In [ ]:
gc.collect()

## Plotting maxes of all components

We will plot the max coefficient case for all components. This helps us see if there is really a difference or not. Confirms the hypothesis a bit.  

You'll need to look closely at the figures. You'll notice that component 7 and 12 don't have particularly strong maximas too.  

In [ ]:
# lets check out the components with the max value of the coefficient for comp=9
for comp_id in range(run.codes.shape[1]):
    print(f"############### comp_id={comp_id} #############################")
    max_idx= np.argmax(np.abs(run.codes[:, comp_id]))
    show_comps_activity_for_case(run.codes, run.components, run.recon, thresholds, max_idx, SHAPE)

# Checking out total-components=27

It has a bunch of warnings. We'll show the figures for all the bad components. The hypothesis that the GMM thresholding is flagging a dead atom is confirmed here for all cases.  

In [ ]:
gc.collect()

In [ ]:
run = c2r[RunId(27,0)]
thresholds = find_thresholds_for_each_component(run.codes)

In [ ]:
bad_comps = [
    0, 7, 9,10,11,12,15,18,20,24,26
]
for c in bad_comps:
    print(f"########################### {c} #####################################################")
    max_idx= np.argmax(np.abs(run.codes[:, c]))
    show_comps_activity_for_case(run.codes, run.components, run.recon, thresholds, max_idx, SHAPE)